# Phase 2: GRPO Alignment on Colab 


In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl" peft accelerate bitsandbytes huggingface_hub
!pip install -q -U datasets

In [ ]:
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

##  Load Dataset


In [ ]:
from datasets import load_dataset
import json as json_module

dataset = load_dataset("myounes21/logos-reasoning-dataset", split="train")

def format_grpo(example):
    prompt = f"<|im_start|>user\n{example['instruction']}<|im_end|>\n<|im_start|>assistant\n"

    ut = example.get('unit_tests', [])
    ut_str = json_module.dumps(ut, ensure_ascii=False) if ut else '[]'

    return {"prompt": prompt, "unit_tests_json": ut_str}

train_dataset = dataset.map(format_grpo)
print(f" Loaded {len(train_dataset)} prompts for GRPO training.")

has_tests = sum(1 for x in train_dataset if x["unit_tests_json"] != "[]")
print(f" {has_tests} examples have unit tests for code execution reward.")
print(f"\nSample prompt:\n{train_dataset[0]['prompt'][:300]}")

##  Reward Functions


In [ ]:
import re
import json as json_module
import signal
import traceback

def extract_code_block(text):
    matches = re.findall(r'```python\n(.*?)```', text, re.DOTALL)
    return matches[-1].strip() if matches else None

def run_unit_tests(code, unit_tests, timeout=3):
    if not code or not unit_tests:
        return None 

    func_names = [f for f in re.findall(r'^def (\w+)\(', code, re.MULTILINE)
                  if not f.startswith('__')]
    if not func_names:
        return 0.0 
    func_name = func_names[-1]

    namespace = {}
    try:
        exec(code, namespace)
    except Exception:
        return 0.0 

    func = namespace.get(func_name)
    if not callable(func):
        return 0.0

    passed = 0
    total = len(unit_tests)

    for test in unit_tests:
        inp = test.get('input', '')
        expected = test.get('expected', '')
        try:
            call_str = f'{func_name}({inp})'
            signal.alarm(timeout)
            result = eval(call_str, namespace)
            signal.alarm(0) 

            if isinstance(expected, str):
                try:
                    expected_val = eval(expected)
                except:
                    expected_val = expected
            else:
                expected_val = expected

            if result == expected_val:
                passed += 1
        except Exception:
            signal.alarm(0) 
            continue

    return 5.0 * (passed / total) if total > 0 else 0.0

def format_reward(completions, **kwargs):
    scores = []
    for c in completions:
        has_think = "<think>" in c and "</think>" in c
        has_code = "```python" in c
        scores.append(1.0 if (has_think and has_code) else 0.0)
    return scores

def correctness_reward(completions, unit_tests_json=None, **kwargs):
    scores = []
    for i, c in enumerate(completions):
        code = extract_code_block(c)

        tests = []
        if unit_tests_json and i < len(unit_tests_json):
            try:
                tests = json_module.loads(unit_tests_json[i]) if isinstance(unit_tests_json[i], str) else unit_tests_json[i]
            except:
                tests = []

        if tests and code:
            score = run_unit_tests(code, tests)
            scores.append(score if score is not None else 0.0)
        elif code and 'def ' in code:
            scores.append(2.0)
        else:
            scores.append(0.0)
    return scores

def language_reward(completions, **kwargs):
    return [2.0 if re.search(r"[\u0600-\u06FF]", c) else 0.0 for c in completions]

def logic_reward(completions, **kwargs):
    keywords = ["إذن", "بالتالي", "لأن", "بما أن", "نستنتج", "أولاً", "ثانياً", "أخيراً"]
    scores = []
    for c in completions:
        count = sum(1 for kw in keywords if kw in c)
        scores.append(min(1.0, count * 0.5)) 
    return scores

test_completion = ["<think>\nإذن نحتاج بالتالي\n</think>\n```python\ndef solve(n):\n    return n * 2\n```"]
test_unit_tests = ['[{"input": "5", "expected": 10}]']

print("Format:", format_reward(test_completion))
print("Correctness:", correctness_reward(test_completion, unit_tests_json=test_unit_tests))
print("Language:", language_reward(test_completion))
print("Logic:", logic_reward(test_completion))

##  Load SFT Adapter as Base Model


In [ ]:
from unsloth import FastLanguageModel
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

max_seq_length = 2048

SFT_ADAPTER_PATH = "/content/drive/MyDrive/logos-sft-adapter-unsloth-final"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = SFT_ADAPTER_PATH, 
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print(f" SFT adapter loaded from: {SFT_ADAPTER_PATH}")
print(f" New GRPO LoRA applied on top.")
!nvidia-smi

##  GRPO Training


In [ ]:
from trl import GRPOConfig, GRPOTrainer
from transformers.trainer_utils import get_last_checkpoint

gc.collect()
torch.cuda.empty_cache()

OUTPUT_DIR = "/content/drive/MyDrive/logos-grpo-adapter"

grpo_config = GRPOConfig(
    output_dir = OUTPUT_DIR,
    learning_rate = 1e-5,
    beta = 0.1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,   
    num_generations = 4,               
    max_prompt_length = 512,
    max_completion_length = 512,       
    num_train_epochs = 1,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    optim = "adamw_8bit",             
    logging_steps = 5,
    save_strategy = "steps",
    save_steps = 25,
    save_total_limit = 3,
    report_to = "none",
    seed = 3407,
)

trainer = GRPOTrainer(
    model = model,
    reward_funcs = [format_reward, correctness_reward, language_reward, logic_reward],
    args = grpo_config,
    train_dataset = train_dataset,
)

checkpoint = get_last_checkpoint(OUTPUT_DIR)

print(" Starting GRPO training...")
if checkpoint is not None:
    print(f" Resuming from checkpoint: {checkpoint}")
    trainer.train(resume_from_checkpoint=checkpoint)
else:
    print(" No checkpoint found. Starting fresh training.")
    trainer.train()

ADAPTER_DIR = "/content/drive/MyDrive/logos-grpo-adapter-final"
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"\n GRPO training complete! Adapter saved to: {ADAPTER_DIR}")

##  Push to HuggingFace Hub (Optional)

In [ ]:
from huggingface_hub import login
login() 

HF_REPO = "myounes21/logos-grpo-adapter" 
model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)
print(f" GRPO adapter pushed to: https://huggingface.co/{HF_REPO}")